# 🏙️ PBL #2 — 데이터 병합 및 전처리 (최종 통합본)
**서울시 청년 비만율 분석 | 하이파이브 팀 | 데이터사이언스개론 26-1**

---

## 📋 이 노트북 한 장으로 끝나는 전처리

원시자료(지역사회건강조사)부터 외부 공공데이터(체육시설·상권)까지 **모든 데이터를 로드·정제·통합**하여 분석용 마스터 테이블을 만듭니다.

| 구분 | 데이터 | 출처 | 범위 |
|------|--------|------|------|
| 원시 | 지역사회건강조사 | 질병관리청 | 2019~2025 (7개년) |
| 외부① | 공공체육시설 통계 | 서울 열린데이터광장 | 2011~2024 |
| 외부② | 등록·신고 체육시설 통계 | 서울 열린데이터광장 | 2010~2024 |
| 외부③ | 상권분석-점포 | 서울 열린데이터광장 | 2021~2025 |
| 외부④ | 상권분석-추정매출 | 서울 열린데이터광장 | 2021~2025 |
| 매핑 | 상권 영역(상권↔자치구) | 서울 열린데이터광장 | - |

## 🎯 변수 설계

- **종속변수**: 비만여부(개인 BMI≥25), 비만율(자치구 %)
- **독립변수(개인)**: 걷기·수면·아침식사·스트레스·흡연·음주·우울
- **독립변수(환경)**: 체육시설 수(H2), 배달친화 외식업 비중·청년 매출비중(H1)
- **통제변수**: 연령·성별·연도

## ⚙️ 전처리에서 해결한 핵심 이슈

1. 연도별 컬럼명 상이(원시자료 2019~21 소문자 / 2022~ 대문자) → 표준화
2. 결측 특수코드(99999·77·88·99 등) → NaN 변환 후 처리
3. 체육시설 결측(중구 2020) → 시계열 선형보간
4. 상권에 자치구 없음 → 영역(DBF)으로 1,650개 상권→25개 자치구 매핑
5. 2025년 상권만 영문 컬럼 → 한글 표준화
6. 비만 이상치 → 생물학적 범위(BMI 10~60)만 제거 (IQR 미사용)

---
# 0. 환경 설정

> 📁 **이 노트북은 모든 데이터 파일과 같은 폴더에 두고 실행합니다.** (PBL#2 폴더에 chs*.zip, 서울시_*.csv, 서울시_상권*.zip 이 함께 있는 구조)

In [ ]:
!pip install pandas numpy pyreadstat dbfread statsmodels matplotlib seaborn

In [ ]:
# 최초 1회만 설치 (주석 해제 후 실행)
# !pip install pandas numpy pyreadstat dbfread statsmodels matplotlib seaborn

import pandas as pd
import numpy as np
import pyreadstat
import glob, zipfile, os, csv
from dbfread import DBF
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'   # Mac은 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')
pd.set_option('display.max_columns', 60)

# ── 경로: 모든 파일이 노트북과 같은 폴더(현재 폴더)에 있음 ──
BASE = './'              # 데이터 파일 위치 = 현재 폴더
WORK = './_work/'        # 압축 해제용 임시 폴더 (자동 생성)
OUT  = './_processed/'   # 결과 저장 폴더 (자동 생성)
for d in [WORK, OUT]:
    os.makedirs(d, exist_ok=True)

# 서울 25개 자치구
GU25 = ['종로구','중구','용산구','성동구','광진구','동대문구','중랑구','성북구','강북구',
        '도봉구','노원구','은평구','서대문구','마포구','양천구','강서구','구로구','금천구',
        '영등포구','동작구','관악구','서초구','강남구','송파구','강동구']
print('✅ 환경 설정 완료 | 작업 폴더:', os.path.abspath(BASE))

: 

---
# PART A. 원시자료 전처리 (지역사회건강조사 → 청년 비만율)

## A-1. SAS 파일 로드 (zip 자동 해제)

`chs19_all.zip` ~ `chs25_all.zip` 7개를 읽습니다. 이미 압축이 풀린 `.sas7bdat`이 있으면 그대로 사용합니다.

In [ ]:
# 연도 ↔ 파일 stem 매핑
CHS = {2019:'chs19_all', 2020:'chs20_all_re', 2021:'chs21_all',
       2022:'chs22_all', 2023:'chs23_all', 2024:'chs24_all', 2025:'chs25_all'}

def get_sas_path(stem):
    """sas7bdat가 이미 있으면 그 경로, 없으면 zip 풀어서 경로 반환"""
    # 1) 현재 폴더 또는 작업폴더에 이미 풀린 파일
    for cand in [BASE+stem+'.sas7bdat', WORK+stem+'.sas7bdat']:
        if os.path.exists(cand):
            return cand
    # 2) zip 해제
    zip_path = BASE+stem+'.zip'
    if not os.path.exists(zip_path):
        return None
    with zipfile.ZipFile(zip_path) as z:
        targets = [n for n in z.namelist() if n.endswith('.sas7bdat') and 'ctv' not in n.lower()]
        if not targets:
            return None
        z.extract(targets[0], WORK)
        return WORK+targets[0]

SAS_PATHS = {}
for y, stem in CHS.items():
    p = get_sas_path(stem)
    if p:
        SAS_PATHS[y] = p
        print(f'  ✅ {y}: {os.path.basename(p)}')
    else:
        print(f'  ⚠️ {y}: 파일 없음 → 건너뜀')
print(f'\n총 {len(SAS_PATHS)}개 연도 준비')

## A-2. 연도별 변수 매핑 (실제 파일로 검증된 표)

연도마다 컬럼명·키몸무게 변수가 달라 표준화합니다.

In [ ]:
VAR_MAP = {
    'age':       {y:'age'         for y in range(2019,2026)},
    'sex':       {y:'sex'         for y in range(2019,2026)},
    'sigungu':   {y:'signgu_code' for y in range(2019,2026)},
    # 시도코드: 2019~2021 소문자, 2022~ 대문자
    'sido':      {2019:'ctprvn_code',2020:'ctprvn_code',2021:'ctprvn_code',
                  2022:'CTPRVN_CODE',2023:'CTPRVN_CODE',2024:'CTPRVN_CODE',2025:'CTPRVN_CODE'},
    # 키·몸무게: 2019 측정값 / 2020 BMI직접 / 2021~ 자가보고
    'height':    {2019:'mea_14z1',2021:'oba_02z1',2022:'oba_02z1',2023:'oba_02z1',
                  2024:'oba_02z1',2025:'oba_02z1'},
    'weight':    {2019:'mea_10z1',2021:'oba_03z1',2022:'oba_03z1',2023:'oba_03z1',
                  2024:'oba_03z1',2025:'oba_03z1'},
    'bmi_given': {2020:'oba_bmi'},
    # 공통 행태변수
    'walk_days': {y:'phb_01z1' for y in range(2019,2026)},
    'breakfast': {y:'nua_01z2' for y in range(2019,2026)},
    'stress':    {y:'mta_01z1' for y in range(2019,2026)},
    'depress':   {y:'mtb_01z1' for y in range(2019,2026)},
    'smoke':     {y:'sma_03z2' for y in range(2019,2026)},
    'drink_freq':{y:'drb_01z3' for y in range(2020,2026)},  # 2019 변수명 상이→제외
    # 수면: 2019 mtc_01z1 / 2023 없음 / 그외 mtc_17z1
    'sleep':     {2019:'mtc_01z1',2020:'mtc_17z1',2021:'mtc_17z1',2022:'mtc_17z1',
                  2024:'mtc_17z1',2025:'mtc_17z1'},
}

# 연도별 실제 컬럼 캐시
YEAR_COLS = {}
for y,p in SAS_PATHS.items():
    _, meta = pyreadstat.read_sas7bdat(p, row_limit=1)
    YEAR_COLS[y] = set(meta.column_names)
print('✅ 변수 매핑 정의 + 연도별 컬럼 캐시 완료')

## A-3. 로드 → 서울 필터 → 청년 필터 → 병합

In [ ]:
def load_year(year):
    needed = {}
    for std, ymap in VAR_MAP.items():
        col = ymap.get(year)
        if col and col in YEAR_COLS[year]:
            needed[col] = std
    df, _ = pyreadstat.read_sas7bdat(SAS_PATHS[year], usecols=list(needed.keys()))
    df = df.rename(columns=needed)
    df['year'] = year
    # 서울 필터 (시도코드 11)
    df['sido'] = df['sido'].astype(str).str.strip().str.replace('.0','',regex=False)
    df = df[df['sido']=='11'].copy()
    return df

frames = [load_year(y) for y in sorted(SAS_PATHS)]
df_raw = pd.concat(frames, ignore_index=True)
print(f'[서울 필터] 병합 {df_raw.shape[0]:,}행')

# 청년 필터 (19~39세)
df = df_raw[(df_raw['age']>=19) & (df_raw['age']<=39)].copy()
print(f'[청년 필터] {df_raw.shape[0]:,} → {df.shape[0]:,}행')
print(df['year'].value_counts().sort_index().to_string())

## A-4. 결측 특수코드 처리 + BMI 계산 + 이상치 처리

- **결측**: 키 `99999`, 빈도 `77/88/99`, 범주 `7/8/9` → NaN
- **이상치**: BMI 10 미만·60 초과만 제거 (비만 분석이라 IQR로 고BMI 자르지 않음)

In [ ]:
# 결측 특수코드 → NaN
for c in ['height','weight','bmi_given']:
    if c in df.columns:
        df[c] = df[c].replace([9999,99999,8888,88888,7777,77777], np.nan)
        df.loc[df[c]>=900, c] = np.nan
for c in ['walk_days','sleep','breakfast','drink_freq']:
    if c in df.columns:
        df[c] = df[c].replace([77,88,99], np.nan)
for c in ['smoke','stress','depress']:
    if c in df.columns:
        df[c] = df[c].replace([7,8,9], np.nan)

# BMI 계산 (키·몸무게 있으면 계산, 2020년은 제공된 BMI 사용)
df['BMI'] = np.nan
m = df['height'].notna() & df['weight'].notna()
df.loc[m,'BMI'] = df.loc[m,'weight'] / (df.loc[m,'height']/100)**2
if 'bmi_given' in df.columns:
    df['BMI'] = df['BMI'].fillna(df['bmi_given'])
df['BMI'] = df['BMI'].round(2)

# 이상치 제거 (생물학적 불가능 범위)
before = len(df)
df = df[(df['BMI']>=10) & (df['BMI']<=60)].copy()
print(f'[이상치 제거] BMI 비현실값: {before:,} → {len(df):,}행')

# 비만여부
df['비만여부'] = (df['BMI']>=25).astype(int)
print(f'청년 비만율: {df["비만여부"].mean()*100:.1f}%, BMI 평균: {df["BMI"].mean():.2f}')

## A-5. 자치구 매핑 + 잔여 결측 처리 + 집계

In [ ]:
# 행정기관코드 → 자치구명
GU_CODE = {
    '11110':'종로구','11140':'중구','11170':'용산구','11200':'성동구','11215':'광진구',
    '11230':'동대문구','11260':'중랑구','11290':'성북구','11305':'강북구','11320':'도봉구',
    '11350':'노원구','11380':'은평구','11410':'서대문구','11440':'마포구','11470':'양천구',
    '11500':'강서구','11530':'구로구','11545':'금천구','11560':'영등포구','11590':'동작구',
    '11620':'관악구','11650':'서초구','11680':'강남구','11710':'송파구','11740':'강동구'}
df['sigungu'] = df['sigungu'].astype(str).str.strip().str.replace('.0','',regex=False)
df['자치구'] = df['sigungu'].map(GU_CODE)
print(f'자치구 매핑: {df["자치구"].nunique()}개, 매핑실패 {df["자치구"].isna().sum()}행')

# 잔여 결측: 빈도형 중앙값, 범주형 최빈값
for v in ['walk_days','sleep','breakfast','drink_freq']:
    if v in df.columns and df[v].notna().any():
        df[v] = df[v].fillna(df[v].median())
for v in ['smoke','stress','depress']:
    if v in df.columns and df[v].notna().any():
        df[v] = df[v].fillna(df[v].mode()[0])

# 개인 수준 저장
keep = ['year','자치구','age','sex','BMI','비만여부','walk_days','sleep',
        'breakfast','stress','smoke','drink_freq','depress']
keep = [c for c in keep if c in df.columns]
df_indiv = df[keep].copy()
df_indiv.to_csv(OUT+'seoul_youth_individual.csv', index=False, encoding='utf-8-sig')

# 자치구×연도 집계
obesity_yr = df_indiv.groupby(['자치구','year']).agg(
    표본수=('비만여부','size'), 비만율=('비만여부','mean'),
    평균BMI=('BMI','mean'), 평균걷기일수=('walk_days','mean'),
    평균수면=('sleep','mean'), 평균아침식사=('breakfast','mean'),
    흡연율=('smoke', lambda x:(x==1).mean())).reset_index()
obesity_yr['비만율'] = (obesity_yr['비만율']*100).round(2)
obesity_yr.to_csv(OUT+'seoul_youth_district_year.csv', index=False, encoding='utf-8-sig')

print(f'\n✅ PART A 완료: 개인 {len(df_indiv):,}행, 자치구×연도 {len(obesity_yr)}행')
print(f'   결측: 개인 {df_indiv.isna().sum().sum()}개')

---
# PART B. 체육시설 통계 전처리 (가설 H2)

## B-1. 통계표 안전 로드 함수

In [ ]:
def read_seoul_stat(path):
    """피벗 통계표 CSV 안전 로드 (따옴표 깨짐 대응)"""
    df = pd.read_csv(path, encoding='utf-8-sig', engine='python', quoting=csv.QUOTE_NONE)
    df = df.dropna(axis=1, how='all')
    df.columns = [str(c).replace('"','').replace(' ','').replace('년','') for c in df.columns]
    for c in df.columns:
        df[c] = df[c].astype(str).str.replace('"','').str.strip()
    return df

def to_num(s):
    return pd.to_numeric(
        s.astype(str).str.replace('-','',regex=False).str.replace(',','',regex=False).replace('',np.nan),
        errors='coerce')

print('✅ 함수 정의')

## B-2. 공공체육시설 (2011~2024) → long 변환

In [ ]:
pub = read_seoul_stat(BASE+'서울시_공공체육시설_통계.csv')
pub = pub.rename(columns={'공공체육시설별':'시설구분','자치구별':'자치구'})
yc = [c for c in pub.columns if c.isdigit()]
pub_sel = pub[(pub['항목']=='개소') & (pub['시설구분']=='합계') & (pub['자치구'].isin(GU25))]
pub_long = pub_sel.melt(id_vars=['자치구'], value_vars=yc, var_name='연도', value_name='공공체육시설수')
pub_long['연도'] = pub_long['연도'].astype(int)
pub_long['공공체육시설수'] = to_num(pub_long['공공체육시설수'])
print(f'공공체육시설: {pub_long.shape}, 연도 {pub_long.연도.min()}~{pub_long.연도.max()}, 자치구 {pub_long.자치구.nunique()}개, 결측 {pub_long.공공체육시설수.isna().sum()}')

## B-3. 등록·신고 체육시설 (2010~2024) + 결측 시계열 보간

**결측 처리**: `-`(미집계) 셀을 자치구별 **선형보간**. 0이면 "시설 없음" 오해, 전체평균이면 그 구 수준 왜곡 → 앞뒤 연도 사이를 자연스럽게 메움.

In [ ]:
reg = read_seoul_stat(BASE+'서울시_등록_신고_체육시설_통계.csv')
reg = reg.rename(columns={'체육시설별':'시설구분','자치구별':'자치구'})
yc2 = [c for c in reg.columns if c.isdigit()]
reg_sel = reg[(reg['항목']=='시설수') & (reg['시설구분']=='체육시설') & (reg['자치구'].isin(GU25))]
reg_long = reg_sel.melt(id_vars=['자치구'], value_vars=yc2, var_name='연도', value_name='민간체육시설수')
reg_long['연도'] = reg_long['연도'].astype(int)
reg_long['민간체육시설수'] = to_num(reg_long['민간체육시설수'])

miss = reg_long[reg_long['민간체육시설수'].isna()]
print(f'보간 전 결측: {len(miss)}개  {miss[["자치구","연도"]].values.tolist() if len(miss) else ""}')
reg_long = reg_long.sort_values(['자치구','연도'])
reg_long['민간체육시설수'] = reg_long.groupby('자치구')['민간체육시설수'].transform(
    lambda s: s.interpolate(method='linear', limit_direction='both'))
print(f'보간 후 결측: {reg_long.민간체육시설수.isna().sum()}개  ✅')

# 두 체육시설 통합
sports = pub_long.merge(reg_long, on=['자치구','연도'], how='outer')
sports['총체육시설수'] = sports['공공체육시설수'].fillna(0) + sports['민간체육시설수'].fillna(0)
sports.to_csv(OUT+'ext_sports.csv', index=False, encoding='utf-8-sig')
print(f'체육시설 통합: {sports.shape}, 연도 {sports.연도.min()}~{sports.연도.max()}')

---
# PART C. 상권 데이터 전처리 (가설 H1)

## C-1. 상권→자치구 매핑 (영역 DBF)

점포·매출 데이터엔 자치구가 없고 상권코드만 있어, 영역 Shapefile의 속성(DBF)에서 매핑을 추출합니다.

In [ ]:
with zipfile.ZipFile(BASE+'서울시_상권분석서비스_영역-상권_.zip') as z:
    z.extractall(WORK+'area')
dbf = glob.glob(WORK+'area/*.dbf')[0]
amap = pd.DataFrame(iter(DBF(dbf, encoding='utf-8')))[['TRDAR_CD','SIGNGU_CD_']]
amap.columns = ['상권_코드','자치구']
amap['상권_코드'] = amap['상권_코드'].astype(str)
print(f'상권→자치구 매핑: {len(amap)}개 상권, {amap.자치구.nunique()}개 자치구')
amap.head()

## C-2. 점포-상권 5개년 집계

배달친화 업종 = 패스트푸드 + 치킨 + 분식. **2025년만 영문 컬럼**이라 표준화합니다.

In [ ]:
FOOD = ['CS100001','CS100002','CS100003','CS100004','CS100005',
        'CS100006','CS100007','CS100008','CS100009','CS100010']
BAEDAL = ['CS100006','CS100007','CS100008']  # 패스트푸드·치킨·분식

ENG2KOR_STORE = {'trdar_cd':'상권_코드','svc_induty_cd':'서비스_업종_코드',
                 'stor_co':'점포_수','frc_stor_co':'프랜차이즈_점포_수'}

def load_csv_flex(path):
    for enc in ['cp949','utf-8-sig','euc-kr']:
        try: return pd.read_csv(path, encoding=enc)
        except Exception: continue
    return None

store_frames = []
for zf in sorted(glob.glob(BASE+'서울시_상권분석서비스_점포-상권*.zip')):
    year = int(''.join(filter(str.isdigit, os.path.basename(zf)[-12:])))
    od = WORK+f'store_{year}'; os.makedirs(od, exist_ok=True)
    with zipfile.ZipFile(zf) as z:
        cf = [n for n in z.namelist() if n.endswith('.csv')][0]; z.extract(cf, od)
    d = load_csv_flex(glob.glob(od+'/*.csv')[0]).rename(columns=ENG2KOR_STORE)
    d['상권_코드'] = d['상권_코드'].astype(str)
    d = d[d['서비스_업종_코드'].isin(FOOD)].merge(amap, on='상권_코드', how='left')
    d['연도'] = year; d['배달친화'] = d['서비스_업종_코드'].isin(BAEDAL)
    store_frames.append(d)
    print(f'  점포 {year}: {len(d):,}행, 자치구결측 {d.자치구.isna().sum()}')
store_all = pd.concat(store_frames, ignore_index=True)

store_gu = store_all.groupby(['자치구','연도']).agg(
    외식업_총점포수=('점포_수','sum'), 프랜차이즈_점포수=('프랜차이즈_점포_수','sum')).reset_index()
bd = store_all[store_all['배달친화']].groupby(['자치구','연도']).agg(배달친화_점포수=('점포_수','sum')).reset_index()
store_gu = store_gu.merge(bd, on=['자치구','연도'], how='left')
store_gu['배달친화_점포비중'] = (store_gu['배달친화_점포수']/store_gu['외식업_총점포수']).round(4)
store_gu.to_csv(OUT+'ext_store.csv', index=False, encoding='utf-8-sig')
print(f'점포 집계: {store_gu.shape}, 연도 {store_gu.연도.min()}~{store_gu.연도.max()}')

## C-3. 추정매출-상권 5개년 집계 (청년 20~30대 매출)

매출 데이터의 연령대별 컬럼으로 **청년(20~30대) 외식 소비**를 직접 측정합니다.

In [ ]:
ENG2KOR_SALES = {'trdar_cd':'상권_코드','svc_induty_cd':'서비스_업종_코드',
                 'thsmon_selng_amt':'당월_매출_금액',
                 'agrde_20_selng_amt':'연령대_20_매출_금액',
                 'agrde_30_selng_amt':'연령대_30_매출_금액'}

sales_frames = []
for zf in sorted(glob.glob(BASE+'서울시_상권분석서비스_추정매출-상권*.zip')):
    year = int(''.join(filter(str.isdigit, os.path.basename(zf)[-12:])))
    od = WORK+f'sales_{year}'; os.makedirs(od, exist_ok=True)
    with zipfile.ZipFile(zf) as z:
        cf = [n for n in z.namelist() if n.endswith('.csv')][0]; z.extract(cf, od)
    d = load_csv_flex(glob.glob(od+'/*.csv')[0]).rename(columns=ENG2KOR_SALES)
    d['상권_코드'] = d['상권_코드'].astype(str)
    d = d[d['서비스_업종_코드'].isin(FOOD)].merge(amap, on='상권_코드', how='left')
    d['연도'] = year
    d['청년매출'] = d['연령대_20_매출_금액'] + d['연령대_30_매출_금액']
    d['배달친화'] = d['서비스_업종_코드'].isin(BAEDAL)
    sales_frames.append(d)
    print(f'  매출 {year}: {len(d):,}행, 자치구결측 {d.자치구.isna().sum()}')
sales_all = pd.concat(sales_frames, ignore_index=True)

sales_gu = sales_all.groupby(['자치구','연도']).agg(
    외식_총매출=('당월_매출_금액','sum'), 외식_청년매출=('청년매출','sum')).reset_index()
bs = sales_all[sales_all['배달친화']].groupby(['자치구','연도']).agg(배달친화_청년매출=('청년매출','sum')).reset_index()
sales_gu = sales_gu.merge(bs, on=['자치구','연도'], how='left')
sales_gu['청년_배달친화_매출비중'] = (sales_gu['배달친화_청년매출']/sales_gu['외식_청년매출']).round(4)
sales_gu.to_csv(OUT+'ext_sales.csv', index=False, encoding='utf-8-sig')
print(f'매출 집계: {sales_gu.shape}, 연도 {sales_gu.연도.min()}~{sales_gu.연도.max()}')

---
# PART D. 마스터 테이블 통합

측정 기간이 다른 데이터를 **자치구별 다년 평균**으로 통일하여 횡단면 마스터를 만듭니다.

In [ ]:
obesity_gu = obesity_yr.groupby('자치구').mean(numeric_only=True).reset_index().drop(columns=['year'])
sports_avg = sports.groupby('자치구')[['공공체육시설수','민간체육시설수','총체육시설수']].mean().reset_index()
store_avg  = store_gu.groupby('자치구')[['외식업_총점포수','배달친화_점포수','배달친화_점포비중']].mean().reset_index()
sales_avg  = sales_gu.groupby('자치구')[['외식_청년매출','배달친화_청년매출','청년_배달친화_매출비중']].mean().reset_index()

master = (obesity_gu
          .merge(sports_avg, on='자치구', how='left')
          .merge(store_avg,  on='자치구', how='left')
          .merge(sales_avg,  on='자치구', how='left'))
master.to_csv(OUT+'master_full.csv', index=False, encoding='utf-8-sig')

print('='*60)
print(f'  ✅ 최종 마스터 테이블 완성: {master.shape}')
print('='*60)
print(f'자치구 수: {master.자치구.nunique()}개 | 결측: {master.isna().sum().sum()}개')
print(f'컬럼: {master.columns.tolist()}')
master.head()

---
# PART E. 가설 검증 (상관 + 회귀)

In [ ]:
from scipy import stats
import statsmodels.api as sm

print('=== 비만율과 외부 변수 상관 (n=25) ===')
for c in ['총체육시설수','공공체육시설수','민간체육시설수','배달친화_점포비중','청년_배달친화_매출비중']:
    r,p = stats.pearsonr(master[c], master['비만율'])
    sig = '***' if p<0.01 else '**' if p<0.05 else '*' if p<0.1 else 'n.s.'
    print(f'  {c:20s}: r={r:+.3f} (p={p:.3f}) {sig}')

X = master[['총체육시설수','청년_배달친화_매출비중','흡연율','평균수면']]
X_std = sm.add_constant((X-X.mean())/X.std())
ols = sm.OLS(master['비만율'], X_std).fit()
print(f'\n=== 다중회귀: R²={ols.rsquared:.3f}, F-p={ols.f_pvalue:.4f} ===')
for n in X.columns:
    print(f'  {n:20s}: β={ols.params[n]:+.3f} (p={ols.pvalues[n]:.3f})')

In [ ]:
# 가설 검증 산점도
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
r_h1,p_h1 = stats.pearsonr(master['청년_배달친화_매출비중'], master['비만율'])
r_h2,p_h2 = stats.pearsonr(master['민간체육시설수'], master['비만율'])

for ax, xcol, mult, color, title, r, p in [
    (axes[0],'청년_배달친화_매출비중',100,'#E76F51',f'H1: 배달친화 외식소비 vs 비만율',r_h1,p_h1),
    (axes[1],'민간체육시설수',1,'#2A9D8F',f'H2: 체육시설 밀도 vs 비만율',r_h2,p_h2)]:
    x = master[xcol]*mult
    ax.scatter(x, master['비만율'], s=90, alpha=0.7, color=color, edgecolor='white')
    z = np.polyfit(x, master['비만율'], 1)
    xl = np.linspace(x.min(), x.max(), 50)
    ax.plot(xl, z[0]*xl+z[1], '--', color='#1D3557', linewidth=1.5)
    for _,row in master.iterrows():
        ax.annotate(row['자치구'], (row[xcol]*mult, row['비만율']), fontsize=8, alpha=0.7,
                    xytext=(4,3), textcoords='offset points')
    sig = '***' if p<0.01 else '**' if p<0.05 else '*' if p<0.1 else ''
    ax.set_title(f'{title} (r={r:+.2f}{sig})', fontweight='bold')
    ax.set_ylabel('청년 비만율 (%)'); ax.grid(alpha=0.3)
axes[0].set_xlabel('청년 배달친화 외식 매출비중 (%)')
axes[1].set_xlabel('민간 체육시설 수 (개)')
plt.tight_layout()
os.makedirs('figures', exist_ok=True)
plt.savefig('figures/hypothesis_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'[H1] r={r_h1:+.3f} (p={p_h1:.3f}) → {"✅ 지지" if r_h1>0 and p_h1<0.05 else "△"}')
print(f'[H2] r={r_h2:+.3f} (p={p_h2:.3f}) → {"✅ 지지" if r_h2<0 and p_h2<0.05 else "△ 약한지지" if r_h2<0 else "❌"}')

---
# 📁 최종 산출물

`_processed/` 폴더에 저장됩니다:

| 파일 | 내용 | 행 수 |
|------|------|-------|
| `seoul_youth_individual.csv` | 청년 개인 (원시자료) | ~45,600 |
| `seoul_youth_district_year.csv` | 자치구×연도 비만율 | 175 |
| `ext_sports.csv` | 자치구×연도 체육시설 | 375 |
| `ext_store.csv` | 자치구×연도 외식 점포 | 125 |
| `ext_sales.csv` | 자치구×연도 청년 외식매출 | 125 |
| **`master_full.csv`** | **통합 마스터 (분석 입력)** | **25** |

## 📋 전처리 요약 (보고서용)

1. **서울·청년 필터링**: 시도코드 11 + 만 19~39세 → 45,621명
2. **연도별 컬럼 표준화**: 2019~21 소문자 / 2022~ 대문자, 키몸무게 변수 통일
3. **결측치**: 특수코드(99999·77·88·99) NaN화 후, 중앙값/최빈값 대체. 체육시설 결측(중구 2020)은 **시계열 선형보간**
4. **이상치**: BMI 생물학적 범위(10~60)만 제거 — 비만 분석이므로 IQR로 고BMI 자르지 않음
5. **공간 통합**: 영역 DBF로 1,650개 상권 → 25개 자치구 매핑
6. **이종 포맷**: 2025년 영문 컬럼 → 한글 표준화
7. **변수 설계**: 단순 개수 대신 비중 사용(상권 규모 통제), 청년 연령대 매출 직접 측정
8. **시점 통일**: 자치구별 다년 평균으로 cross-sectional 구성

---
*하이파이브 팀 | 데이터사이언스개론 PBL #2 | 26-1*